# 01 · The international football network

A different lens on 150 years of results: treat every nation as a **node** and every
match as an **edge**, and ask what the *shape* of that graph says about how hard it is
to rank national teams.

**Thesis.** National teams overwhelmingly play *within* their own confederation, so the
match graph is clustered by continent with only thin "bridge" games linking the blocs.
That sparse cross-continental connectivity is *why* ranking across confederations — and
seeding a 48-team World Cup — is genuinely hard.

This notebook builds the graph, quantifies the structure, and exports
`site/data/network.json` for the interactive D3 page. It's the analysis; the page is the
showroom.

In [1]:
import json

import pandas as pd

from mlfootball import data, network as net

pd.set_option("display.max_rows", 20)

df = data.load()
print(f"{len(df):,} internationals, {df.date.min().date()} → {df.date.max().date()}")

49,291 internationals, 1872-11-30 → 2026-06-01


In [2]:
# Build both graphs: undirected "who-plays-whom" (structure) and directed "beat-graph"
# (ranking). Restricted to the modern era with an iterative min-matches filter.
Gu = net.build_undirected(df)
Gd = net.build_directed(df)
print(f"Graph: {Gu.number_of_nodes()} teams, {Gu.number_of_edges()} edges")

Graph: 217 teams, 5442 edges


## Finding 1 — national teams almost never leave home

The single most important number: what share of games stay *within* a confederation?

In [3]:
rep = net.connectivity_report(Gu)
print(f"WITHIN-confederation: {rep['within_share']:.1%}")
print(f"CROSS-confederation:  {rep['cross_share']:.1%}\n")
pd.Series(rep["insularity"], name="insularity").sort_values(ascending=False).map("{:.1%}".format)

WITHIN-confederation: 86.3%
CROSS-confederation:  13.7%



CAF         90.8%
UEFA        90.2%
AFC         85.5%
CONCACAF    84.2%
OFC         72.9%
CONMEBOL    61.1%
Name: insularity, dtype: str

~86% of games never cross a continent. Big self-sufficient blocs (CAF, UEFA) barely look
outward; CONMEBOL — just ten teams — is forced to. This is the structural reason the
forecast is least certain about cross-continental match-ups: the evidence linking them is thin.

## Finding 2 — the continents draw themselves

Can an algorithm that knows nothing about geography recover the confederations from match
patterns alone? Run Louvain community detection and compare to the real partition.

In [4]:
com = net.communities(Gu)
print(f"Louvain communities: {com['n_communities']}  (vs 6 real confederations)")
print(f"NMI vs confederations: {com['nmi']:.3f}")
print(f"Modularity — confed {com['modularity_confed']:.3f} · Louvain {com['modularity_louvain']:.3f}")

# Which confederations landed in each detected community?
from collections import Counter
conf = {n: Gu.nodes[n]["confederation"] for n in Gu}
comm_of = com["node_comm"]
mix = {}
for n, c in comm_of.items():
    mix.setdefault(c, Counter())[conf[n]] += 1
for c, cnt in sorted(mix.items()):
    print(f"  community {c}: " + ", ".join(f"{k}:{v}" for k, v in cnt.most_common()))

Louvain communities: 5  (vs 6 real confederations)
NMI vs confederations: 0.963
Modularity — confed 0.635 · Louvain 0.636
  community 0: CAF:57
  community 1: AFC:47
  community 2: CONCACAF:41, CONMEBOL:10
  community 3: UEFA:55
  community 4: OFC:7


Almost perfect (NMI 0.96) — but **five blocs, not six**: CONMEBOL and CONCACAF fuse into a
single "Americas" community. The ten South American sides play North/Central America so often
(qualifiers, Copa América) that the graph can't separate them.

## Finding 3 — a ranking for free, and where it's shaky

In [5]:
pr = net.pagerank_ranking(Gd)
pr.head(15)

,team,pagerank,confederation,rank
0,Brazil,0.023754,CONMEBOL,1
1,Argentina,0.021472,CONMEBOL,2
2,France,0.021059,UEFA,3
3,Spain,0.020652,UEFA,4
4,Germany,0.017725,UEFA,5
5,Colombia,0.016350,CONMEBOL,6
6,Mexico,0.015301,CONCACAF,7
7,Portugal,0.015273,UEFA,8
8,Netherlands,0.015258,UEFA,9
9,England,0.015088,UEFA,10


The graph hands you a power ranking via PageRank on the beat-graph. The teams it's *least*
sure about are the poorly-connected ones — exactly the teams the global match graph barely
reaches. Those are the bridge teams that hold the whole ranking together:

In [6]:
print("Bridge teams (most cross-confederational, well-connected):")
print(", ".join(net.bridge_teams(Gu)))

Bridge teams (most cross-confederational, well-connected):
New Zealand, Mexico, Brazil, Uruguay, Japan, United States, South Korea, Argentina, Ecuador, Chile


## Export for the D3 page

Ship nodes + links + metrics; the page force-directs the layout in-browser so it stays
interactive (drag, hover, filter) rather than a baked image.

In [7]:
export = net.to_graph_export(Gu, Gd)
out_path = net.DATA_DIR.parent / "site" / "data" / "network.json"
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(export, separators=(",", ":")))
print(f"Wrote {out_path.relative_to(net.DATA_DIR.parent)}  "
      f"({len(export['nodes'])} nodes, {len(export['links'])} links, "
      f"{out_path.stat().st_size/1024:.0f} KB)")

Wrote site/data/network.json  (217 nodes, 3840 links, 289 KB)
